# Case 3: 3D Full-Body Dyadic (Mirror Game) — reproduction

Two participants perform a mirror game under three visual-coupling conditions —
**back-to-back** (no visual info), **unidirectional** (follower sees leader), and
**face-to-face** (mutual). Recorded in 3-D with ZED stereo cameras (38 keypoints).

We reproduce the Case-3 finding: visual coupling increases whole-body movement
magnitude (acceleration) and leader–follower coordination, from a five-keypoint
subset (head + both wrists + both ankles).

Workflow:
1. **Load & resample** a trial (the ZED stream is variable-rate → uniform 30 Hz).
2. **Choose embedding parameters** `(τ, m)` from AMI/FNN evidence — you commit.
3. **Run the reproduction** across all dyads and trials.
4. **Reproduce the figure.**

All analysis logic lives in the library; this notebook only calls it.

## Configuration

Point these at your Mirror-Game data and the conditions table, and set the run
toggle.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "/Volumes/X9_Pro/Case_Study_3_Mirror_Game"
from pose_dynamics.case_studies.mirror_game import default_conditions_csv
CONDITIONS_CSV = default_conditions_csv()

RUN_FULL   = False           # False = a few pairs (fast); True = all 18 dyads
SUBSET_PAIRS = [1, 2, 3]     # pairs analyzed when RUN_FULL is False

## 1. Load and resample a trial

`load_and_resample` reads a ZED 3-D export (`timestamp_ns, dt_ms, x0,y0,z0, ...`)
and resamples it from its variable frame rate onto a uniform 30 Hz grid using the
timestamps.

In [ ]:
from pose_dynamics.case_studies.mirror_game import load_and_resample, parse_file

sample_file = next(p for p in Path(DATA_DIR).glob("P001_T1_P1_pose_3d.csv") if not p.name.startswith("._"))
seq = load_and_resample(sample_file)
print(seq.summary())
seq.plot_coverage();

## 2. Choosing the embedding parameters (human-in-the-loop)

The CRQA signal is each keypoint's 3-D magnitude time series. We estimate `(τ, m)`
from AMI/FNN across these signals pooled over a few trials, then commit.

In [ ]:
from pose_dynamics.preprocessing import butterworth_filter
from pose_dynamics.features import FeaturePipeline
from pose_dynamics.embedding import (select_embedding, Signal, select_embedding,
                                     plot_embedding_evidence)
from pose_dynamics.case_studies.mirror_game import config as C

# build per-keypoint magnitude signals from a few trials of pair 1
signals = []
for f in sorted(p for p in Path(DATA_DIR).glob("P001_T*_P1_pose_3d.csv") if not p.name.startswith("._"))[:6]:
    s = load_and_resample(f)
    s = butterworth_filter(FeaturePipeline.from_config(
        [{"primitive": "center", "params": {"reference": C.PELVIS}},
         {"primitive": "select_keypoints", "params": {"indices": C.SUBSET_INDICES, "names": C.SUBSET_NAMES}}]
        ).run(s).pose, cutoff_hz=C.FILTER_CUTOFF, order=C.FILTER_ORDER)
    for k, name in enumerate(C.SUBSET_NAMES):
        mag = np.linalg.norm(s.coords[:, k, :], axis=1)
        signals.append(Signal(f"{f.stem}_{name}", mag, group={"keypoint": name}))

evidence = select_embedding(signals, tau_grid=(10, 25), m_grid=(3, 6),
                            ami_max_lag=40, fnn_max_dim=8, subset=40, seed=0)
plot_embedding_evidence(evidence)
print(evidence.justification)

### Commit

Consistent with the paper we commit **τ = 20, m = 4** (the values in the Case-3
config).

In [ ]:
params = evidence.commit(tau=20, m=4, notes="Case 3: committed from AMI/FNN")
params.to_dict()

## 3. Run the reproduction

`run_reproduction` pairs the two participants for each dyad × trial, resamples,
trims to their overlapping window, centres on the pelvis, filters (5 Hz), keeps the
five-keypoint subset, and per keypoint runs **cross-RQA on the magnitude time
series** — averaging across the five keypoints. Acceleration RMS is the kinematic
summary.

**Recurrence mode.** These settings target a **fixed 2.5% recurrence rate**, so
%REC is pinned (a convergence check) and the *achieved radius* is the informative
density measure: a smaller radius means the two bodies' states sit closer in phase
space — more coordination.

In [ ]:
from pose_dynamics.case_studies.mirror_game import run_reproduction

pairs = None if RUN_FULL else SUBSET_PAIRS
df = run_reproduction(DATA_DIR, CONDITIONS_CSV, pairs=pairs, progress=True)
print(f"{len(df)} dyad-trials")
df.groupby("condition", observed=True)[["accel_rms", "cross_perc_recur", "cross_radius", "cross_lmax"]].mean()

In [ ]:
df.to_csv("mirror_case3_results.csv", index=False)

## 4. Reproduce the figure

Group-averaged (mean ± SEM) by visual-coupling condition: acceleration RMS, the
cross-recurrence radius (at 2.5% REC), and the maximum diagonal line length.

In [ ]:
from pose_dynamics.case_studies.mirror_game import plot_case3_figure
plot_case3_figure(df);

## 5. Inferential statistics (dataset-specific)

The package deliberately holds **no inferential statistics** — it emits the tidy
results table. The models below are specific to *this* dataset and live only in the
notebook. Linear mixed-effects models with a fixed effect of visual-coupling
condition (back-to-back as reference) and a random intercept for pair give the β
coefficients (uni, f2f vs. b2b) the paper's coefficient figure reports.

In [ ]:
import statsmodels.formula.api as smf
import pandas as pd
import warnings; warnings.simplefilter("ignore")

coef_rows = []
for metric in ["accel_rms", "cross_radius", "cross_lmax"]:
    m = smf.mixedlm(f"{metric} ~ C(condition, Treatment('b2b'))",
                    df, groups=df["pair"]).fit()
    for term, label in [("C(condition, Treatment('b2b'))[T.uni]", "uni"),
                        ("C(condition, Treatment('b2b'))[T.f2f]", "f2f")]:
        coef_rows.append({"metric": metric, "vs_b2b": label,
                          "beta": m.params[term], "SE": m.bse[term], "p": m.pvalues[term]})
pd.DataFrame(coef_rows).round(4)

## 6. Principal-Movements (PCA) diagnostic

A global PCA on the centred, Procrustes-aligned, filtered 38-keypoint 3-D poses
decomposes whole-body motion into **Principal Movements (PMs)**. This is used only
as a data-quality / structure diagnostic — the recurrence analyses use the
five-keypoint subset, not PC scores. The paper reports the first ~14 PMs capturing
~96% of postural variance.

In [ ]:
from pose_dynamics.case_studies.mirror_game import run_pca_diagnostic, plot_variance, plot_principal_movements

# build the file list for the PCA fit (a subset of pairs gives a stable decomposition)
all_files = [str(p) for p in Path(DATA_DIR).glob("P*_T*_P*_pose_3d.csv") if not p.name.startswith("._")]
pca_files = all_files if RUN_FULL else [f for f in all_files if any(f"P00{p}_" in f for p in (1, 2, 3, 4))]
model, n_kp = run_pca_diagnostic(pca_files, n_components=20, progress=False)
print(f"{model.n_components_for(0.96)} PMs reach 96% variance; "
      f"14 PMs = {model.cumulative_variance()[13]*100:.1f}%")
plot_variance(model);

In [ ]:
plot_principal_movements(model, n_kp, n_pms=14);

## Notes

- **Alignment is not needed for this figure.** The CRQA signal (per-keypoint 3-D
  magnitude) and the kinematics are invariant to the rigid per-trial Procrustes
  rotation, so only pelvis centring matters here. Procrustes/canonicalisation
  matter for the Principal-Movements PCA diagnostic (not reproduced here).
- **What reproduces is the pattern:** acceleration rises with visual coupling, and
  coordination increases (radius shrinks, line length grows) from back-to-back to
  the coupled conditions — matching the paper.
- **Fixed radius instead?** To reproduce the paper's *%REC-varies* figure, set the
  CRQA params to `radius_mode="fixed_radius"`; then %REC is the outcome.
- Stats require `statsmodels` (`pip install statsmodels`) — a notebook dependency,
  not part of the core package.